# XGBoost Model for PL-S5E6

Feature Engineering idea : [EDA Notebook](https://www.kaggle.com/code/suhyukchoi/pl-s5e6-eda-notebook)

In [ ]:
VER = 2

# Setup for Training

## Loading Libraries

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from itertools import combinations
import gc
import optuna

## Loading Datasets

In [ ]:
train = pd.read_csv("data/train.csv", index_col = 0)
test = pd.read_csv("data/test.csv", index_col = 0)
org_train = pd.read_csv("data/Fertilizer Prediction.csv")

train = pd.concat([train, org_train], axis=0, ignore_index=True)

train.head()


## Check Dataset

### Train Dataset

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
test.info()

In [ ]:
test.describe()

- No Missing Values. Don't need to impute.

## Define Useful Methods/Variables

### Variables

In [ ]:
TARGET = 'Fertilizer Name'
COLUMNS = train.columns.tolist()
QUAN_COLUMNS = [col for col in train.columns if train[col].dtype != 'object' and col != TARGET]
CAT_COLUMNS = [col for col in train.columns if train[col].dtype == 'object' and col != TARGET]

print("Total Columns:" ,COLUMNS)
print('Target column:', TARGET)
print('Quantitative columns:', QUAN_COLUMNS)
print('Categorical columns:', CAT_COLUMNS)

### Methods

In [ ]:
def FE_for_gbdt(train, test):
    dfs = [train, test]
    
    # 1. Categorical Columns Encoding.
    
    for df in dfs:
        new_CAT_COLUMNS = set() # For Uniqueness.
        '''
        1.  Add New Cateogical Column Here!
            You need to add new categorical columns into new_CAT_COLUMNS list to successfully conduct Interaction terms.
        '''
        # ========================================
        
        df["Soil Type_Crop Type"] = df["Soil Type"].astype(str) + "_" + df["Crop Type"].astype(str)
        new_CAT_COLUMNS.add("Soil Type_Crop Type")

        # =========================================
        '''
        2.  Add 2-level Interaction term between Categorical Columns.
            Automatically conducted.
        

        comb = list(combinations(CAT_COLUMNS + list(new_CAT_COLUMNS) + ["Temparature"], 2))
        
        for c1, c2 in comb:
            new_col = f"{c1}_{c2}"
            df[new_col] = df[c1].astype(str) + "_" + df[c2].astype(str)
            new_CAT_COLUMNS.add(new_col)
        '''
    # Update CAT_COLUMNS.
    CAT_COLUMNS.extend(list(new_CAT_COLUMNS))

    # Ordinal Encoding for Categorical Columns.
    for col in CAT_COLUMNS:
        le = OrdinalEncoder()
        train[col] = le.fit_transform(pd.DataFrame(train[col].astype(str)))
        test[col] = le.transform(pd.DataFrame(test[col].astype(str)))

    # 2. Quantitative Columns Encoding.
    for df in dfs:
        new_QUAN_COLUMNS = set()  # For Uniqueness.
        new_RATIO_COLUMNS = set()  # To avoid duplicate columns in interaction feature step (ex : (c1/c2) * c2 == c1), we need to specify ratio columns.
        
        '''
        1.  Add New Quantitative Column Here!
            You need to add new quantitative columns into new_QUAN_COLUMNS list to successfully conduct Interaction terms.
        
        # ========================================
        # Add Total_Nutrients Feature.
        df["Total_Nutrients"] = df["Nitrogen"] + df["Phosphorous"] + df["Potassium"]
        new_QUAN_COLUMNS.add("Total_Nutrients")
        
        # Add Nutrients Ratio Feature.
        comb = list(combinations(["Nitrogen", "Phosphorous", "Potassium"], 2))
        
        for c1, c2 in comb:
            new_col = f"{c1}/{c2}"
            k = 1 # Smoothing Factor.
            df[new_col] = df[c1] / (df[c2] + k)  # Avoid division by zero.
            new_QUAN_COLUMNS.add(new_col)
            new_RATIO_COLUMNS.add(new_col)
        
        # Add Humidity to Moisture Ratio Feature.
        k = 1 # smoothing factor 
        df['Humidity/Moisture'] = df['Humidity'] / (df['Moisture'] + k)  # Avoid division by zero
        new_QUAN_COLUMNS.add("Humidity/Moisture")
        new_RATIO_COLUMNS.add("Humidity/Moisture")
        
        # =========================================
        
        2.  Add 2-level Interaction term between Quantitative Columns.
            Automatically conducted.
        
        comb = list(combinations(QUAN_COLUMNS + list(new_QUAN_COLUMNS), 2))
        
        for c1, c2 in comb:
            # Check ratio columns to avoid duplicate columns.
            if c1 in new_RATIO_COLUMNS:
                if c1.split('/')[1] == c2:
                    continue
            elif c2 in new_RATIO_COLUMNS:
                if c2.split('/')[1] == c1:
                    continue
            else:
                new_col = f"{c1}*{c2}"
                df[new_col] = df[c1] * df[c2]
                new_QUAN_COLUMNS.add(new_col)
          '''  
    # Update QUAN_COLUMNS.
    QUAN_COLUMNS.extend(list(new_QUAN_COLUMNS))

    # 3. Target Label Encoding.
    le_target = LabelEncoder()
    train[TARGET] = le_target.fit_transform(train[TARGET])
    
    # Delete Unnecessary Variables for memory Efficiency.
    del new_CAT_COLUMNS, new_QUAN_COLUMNS, new_RATIO_COLUMNS, dfs
    gc.collect()
    
    return train, test, le_target

In [ ]:
def prob_to_top_k_label(prob, k = 3):
    return np.argsort(prob, axis = 1)[:, -k:][:, ::-1]

In [ ]:
import numpy as np

def MAP3_score(y_true, y_pred, k=3):
    """
    Compute Mean Average Precision at k (MAP@3)
    
    y_pred : 2D array of shape (n_samples, k)
             Each row contains predicted class indices (Top-k)
    y_true : 1D array of shape (n_samples,)
             Ground truth class indices
    k      : Number of top predictions to consider
    """
    if y_pred.shape[1] != k:
        raise ValueError(f"y_pred must have shape (n_samples, {k})")
        
    weights = np.array([1.0 / (i + 1) for i in range(k)])  # MAP ??? ? ???? ??
    scores = np.sum((y_pred == y_true.reshape(-1, 1)) * weights, axis=1)
    return np.mean(scores)


In [ ]:
def make_sub(top_k_preds, le_target, k = 3):
    # Load Sample Submission
    sample_submission = pd.read_csv("data/sample_submission.csv")
    org_y_pred = list(top_k_preds.copy())
    
    # Convert top_k_preds to original target labels
    for row in range(top_k_preds.shape[0]):
        org_y_pred[row] = le_target.inverse_transform(list(top_k_preds[row]))
    sample_submission[TARGET] = [' '.join(x) for x in org_y_pred]

    # Save submission file
    sample_submission.to_csv(f'submission/sub_xgb_{VER}.csv', index = False)

    # Return the sample submission DataFrame
    return sample_submission

# Tuning XGBoost Parameter 

- Hyperparameter tuning using `optuna`

In [ ]:
# Setting up the data.
train, test, le_target = FE_for_gbdt(train, test)
train_X = train.drop(TARGET, axis=1)
train_y = train[TARGET]

In [ ]:
train.head()

## Parameter Setting

Using 5 folds.

- `max_depth` : 5 ~ 10
- `learning_rate` : 0.005 ~ 0.01
- `subsample` : 0.7 ~ 1.0
- `colsample_bytree` : 0.7 ~ 1.0
- `reg_alpha` : 0 ~ 1
- `reg_lambda` : 0 ~ 1

In [ ]:
def objective(trial):
    xgb_params = {
        'objective' : 'multi:softprob',
        'num_class' : len(le_target.classes_),
        'eval_metric' : 'mlogloss',
        'device' : 'cuda',
        'tree_method' : 'hist',
        'max_depth' : trial.suggest_int('max_depth', 5, 10),
        'learning_rate' : trial.suggest_float('learning_rate', 0.01, 0.05),
        'subsample' : trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha' : trial.suggest_float('reg_alpha', 0.5, 3.0),
        'reg_lambda' : trial.suggest_float('reg_lambda', 0.5, 3.0),
        'n_estimators' : 10000,
        'early_stopping_rounds' : 100,
        'random_state' : 42,
    }

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_probs = np.zeros((train_X.shape[0], len(le_target.classes_)))
    y_pred_probs = np.zeros((test.shape[0], len(le_target.classes_)))

    for fold, (train_idx, val_idx) in enumerate(kf.split(train_X, train_y)):

        X_train, X_val = train_X.iloc[train_idx], train_X.iloc[val_idx]
        y_train, y_val = train_y.iloc[train_idx], train_y.iloc[val_idx]

        model = xgb.XGBClassifier(**xgb_params)
        model.fit(X_train, y_train, 
                  eval_set=[(X_val, y_val)], 
                  verbose=1000)
        oof_probs[val_idx] = model.predict_proba(X_val)
        y_pred_probs += model.predict_proba(test) / kf.n_splits
        del model, X_train, X_val, y_train, y_val
        gc.collect()

    oof_score = MAP3_score(train_y.values, prob_to_top_k_label(oof_probs))
    print(f"OOF MAP@3 Score: {oof_score:.4f}")

    return oof_score

In [ ]:
# Conduct hyperparameter optimization using Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [ ]:
# Best Parameters
'''
best_params = study.best_params

for key, value in {'objective': 'multi:softprob',
                  'num_class': len(le_target.classes_),
                  'eval_metric': 'mlogloss',
                  'device': 'cuda',
                  'tree_method': 'hist',
                  'n_estimators': 100000,
                  'early_stopping_rounds': 100,
                  'random_state': 42}.items():
    best_params[key] = value

'''

best_params = {
        'objective' : 'multi:softprob',
        'num_class' : len(le_target.classes_),
        'eval_metric' : 'mlogloss',
        'device' : 'cuda',
        'tree_method' : 'hist',
        'max_depth' : 10,
        'learning_rate' : 0.024450161322014278,
        'subsample' : 0.6924178877102732,
        'colsample_bytree' : 0.4046732451199841,
        'reg_alpha' : 0.5295976260675789,
        'reg_lambda' : 2.211372813069924,
        'n_estimators' : 100000,
        'early_stopping_rounds' : 100,
        'random_state' : 42,
    }

print("Best Parameters:", best_params)

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros((train_X.shape[0], len(le_target.classes_)))
y_pred_probs = np.zeros((test.shape[0], len(le_target.classes_)))

for fold, (train_idx, val_idx) in enumerate(kf.split(train_X, train_y)):

    X_train, X_val = train_X.iloc[train_idx], train_X.iloc[val_idx]
    y_train, y_val = train_y.iloc[train_idx], train_y.iloc[val_idx]

    model = xgb.XGBClassifier(**best_params)
    model.fit(X_train, y_train, 
                  eval_set=[(X_val, y_val)], 
                  verbose=1000)
    oof_probs[val_idx] = model.predict_proba(X_val)
    y_pred_probs += model.predict_proba(test) / kf.n_splits

oof_score = MAP3_score(train_y.values, prob_to_top_k_label(oof_probs))
print(f"OOF MAP@3 Score: {oof_score:.4f}")

y_pred_label = prob_to_top_k_label(y_pred_probs)

In [ ]:
sub = make_sub(y_pred_label, le_target)

In [ ]:
# Save oof files
np.save(f'oof/oof_xgb_{VER}.npy', oof_probs)

In [ ]:
# Save Parameters
import json
with open(f'parameter/xgb_params_{VER}.json', 'w') as f:
    json.dump(best_params, f)